# WIA3003: Academic Project II
## QCNN Feature Ablation

Compare all eight measurements with orbital-only and nasal-only inputs. Retain eight wires and set excluded angles to zero after scaling. Refit the circuit and sigmoid readout for each feature set. Use the saved 664 fitting and 168 validation records; exclude the 210 test records. Codex assisted with preparation.

## 1.0 Dataset Preparation

In [1]:
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
from pennylane import numpy as pnp
from sklearn.preprocessing import MinMaxScaler

current_folder = Path.cwd()
project_folder = (
    current_folder.parent
    if current_folder.name == "notebooks"
    else current_folder
)

split_path = project_folder / "results" / "qcnn_baseline" / "run_details.json"
split_details = json.loads(split_path.read_text(encoding="utf-8"))

data_path = (
    Path.home()
    / "Downloads"
    / "Orbital, Nasal and Craniofacial Class Pattern.xls"
)

workbook_hash = hashlib.sha256(data_path.read_bytes()).hexdigest()
if workbook_hash != split_details["dataset_sha256"]:
    raise ValueError(
        "The workbook differs from the one used for the saved QCNN split. "
        "Check the file before reusing the row positions."
    )

df = pd.read_excel(data_path, sheet_name="Raw Data")
df.columns = df.columns.str.replace(r"\s+", " ", regex=True).str.strip()


In [2]:
features = split_details["features"]
label_mapping = split_details["label_mapping"]

fit_indices = np.array(split_details["fit_row_positions"], dtype=int)
val_indices = np.array(split_details["validation_row_positions"], dtype=int)

fit_records = df.iloc[fit_indices]
val_records = df.iloc[val_indices]

X_fit_raw = fit_records[features].apply(pd.to_numeric, errors="raise")
X_val_raw = val_records[features].apply(pd.to_numeric, errors="raise")

y_fit_raw = fit_records["Gender"].astype("string").str.strip().map(label_mapping)
y_val_raw = val_records["Gender"].astype("string").str.strip().map(label_mapping)

assert X_fit_raw.shape == (664, 8)
assert X_val_raw.shape == (168, 8)
assert not set(fit_indices).intersection(val_indices)
assert np.isfinite(X_fit_raw.to_numpy()).all()
assert np.isfinite(X_val_raw.to_numpy()).all()
assert not y_fit_raw.isna().any()
assert not y_val_raw.isna().any()

fit_groups = set(X_fit_raw.itertuples(index=False, name=None))
val_groups = set(X_val_raw.itertuples(index=False, name=None))
shared_groups = fit_groups & val_groups

assert not shared_groups, "Identical measurements appear in both partitions."

print("Fitting:", X_fit_raw.shape)
print("Validation:", X_val_raw.shape)
print("Shared measurement groups:", len(shared_groups))


Fitting: (664, 8)
Validation: (168, 8)
Shared measurement groups: 0


In [3]:
scaler = MinMaxScaler(feature_range=(0, np.pi), clip=True)

X_fit = pnp.array(
    scaler.fit_transform(X_fit_raw),
    requires_grad=False,
)

X_val = pnp.array(
    scaler.transform(X_val_raw),
    requires_grad=False,
)

y_fit = pnp.array(y_fit_raw.to_numpy(dtype=int), requires_grad=False)
y_val = pnp.array(y_val_raw.to_numpy(dtype=int), requires_grad=False)

validation_values = X_val_raw.to_numpy()
clipped_count = (
    (validation_values < scaler.data_min_)
    | (validation_values > scaler.data_max_)
).sum()

print("Validation feature values clipped:", int(clipped_count))


Validation feature values clipped: 1


## 2.0 Circuit and Readout

The circuit still returns four expectations. Apply a sigmoid to a weighted sum of them. Initialise the logit to four times the old score minus its 0.5 offset, i.e. 4 × (old score − 0.5). This preserves the initial decision boundary and matches the slope at the boundary, but initial probabilities differ away from it. Compute binary cross-entropy directly from logits for numerical stability.

In [4]:
import time
import pennylane as qml
from sklearn.metrics import accuracy_score, f1_score, log_loss, recall_score

SEEDS = [42, 123, 2026]
BATCH_SIZE = 32
MAX_EPOCHS = 30
PATIENCE = 5
RATE = 0.01


In [5]:
n_qubits = 8
surviving_wires = [1, 3, 5, 7]

def convolution_block(parameters, first, second):
    qml.Rot(*parameters[:3], wires=first)
    qml.Rot(*parameters[3:], wires=second)
    qml.CNOT(wires=[first, second])

def convolution_layer(parameters):
    # Apply the same block across neighbouring pairs.
    for first in [0, 2, 4, 6]:
        convolution_block(parameters, first, first + 1)
    for first in [1, 3, 5]:
        convolution_block(parameters, first, first + 1)

def pooling_layer(angle):
    for source, target in [(0, 1), (2, 3), (4, 5), (6, 7)]:
        qml.CRY(angle, wires=[source, target])

device = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(device, interface="autograd", diff_method="backprop")
def qcnn_circuit(inputs, weights):
    for wire in range(n_qubits):
        qml.RY(inputs[..., wire], wires=wire)
    convolution_layer(weights[:6])
    convolution_layer(weights[6:12])
    pooling_layer(weights[12])
    return [qml.expval(qml.PauliZ(wire)) for wire in surviving_wires]




In [6]:
def readout_logit(parameters, inputs):
    z = pnp.stack(qcnn_circuit(inputs, parameters[:13]))
    return pnp.sum(parameters[13:17, None] * z, axis=0) + parameters[17]


def predict_probability(parameters, inputs, readout):
    logits = readout_logit(parameters, inputs)
    return pnp.exp(-pnp.logaddexp(0, -logits))


def binary_loss(parameters, inputs, labels, readout):
    logits = readout_logit(parameters, inputs)
    return pnp.mean(pnp.logaddexp(0, (1 - 2 * labels) * logits))


In [7]:
def train_readout(seed, readout, learning_rate, X_fit, X_val):
    # Same circuit initialisation for both readouts at each seed.
    rng = np.random.default_rng(seed)
    circuit_weights = rng.uniform(-0.1, 0.1, size=13)

    if readout == "Learned readout":
        initial_weights = np.concatenate(
            [circuit_weights, np.full(4, -0.5), [0.0]]
        )
    else:
        initial_weights = circuit_weights

    parameters = pnp.array(initial_weights, requires_grad=True)
    optimiser = qml.AdamOptimizer(stepsize=learning_rate)
    shuffle_rng = np.random.default_rng(seed)

    best_loss = np.inf
    best_epoch = None
    best_parameters = None
    waiting = 0
    history = []
    start = time.perf_counter()

    for epoch in range(1, MAX_EPOCHS + 1):
        order = shuffle_rng.permutation(len(X_fit))

        for offset in range(0, len(order), BATCH_SIZE):
            batch = order[offset : offset + BATCH_SIZE]

            def batch_loss(current):
                return binary_loss(
                    current, X_fit[batch], y_fit[batch], readout
                )

            parameters = optimiser.step(batch_loss, parameters)

        fitting_loss = float(binary_loss(parameters, X_fit, y_fit, readout))
        validation_loss = float(binary_loss(parameters, X_val, y_val, readout))

        if not np.isfinite([fitting_loss, validation_loss]).all():
            raise ValueError(f"Non-finite loss: {readout}, seed {seed}, epoch {epoch}")

        history.append({
            "Readout": readout,
            "Seed": seed,
            "Epoch": epoch,
            "Fitting loss": fitting_loss,
            "Validation loss": validation_loss,
        })

        if validation_loss < best_loss:
            best_loss = validation_loss
            best_epoch = epoch
            best_parameters = np.array(parameters, dtype=float, copy=True)
            waiting = 0
        else:
            waiting += 1
            if waiting >= PATIENCE:
                break

    runtime = time.perf_counter() - start

    # Score the best checkpoint, not the last training epoch.
    checkpoint = pnp.array(best_parameters, requires_grad=False)
    probabilities = np.asarray(
        predict_probability(checkpoint, X_val, readout), dtype=float
    )
    predictions = (probabilities >= 0.5).astype(int)
    labels = np.asarray(y_val, dtype=int)

    result = {
        "Readout": readout,
        "Seed": seed,
        "Best epoch": best_epoch,
        "Epochs run": len(history),
        "Validation macro F1": f1_score(
            labels, predictions, average="macro", zero_division=0
        ),
        "Validation accuracy": accuracy_score(labels, predictions),
        "Validation loss": log_loss(labels, probabilities, labels=[0, 1]),
        "Runtime (s)": runtime,
        "Female recall": recall_score(labels, predictions, pos_label=0, zero_division=0),
        "Male recall": recall_score(labels, predictions, pos_label=1, zero_division=0),
    }

    return result, history, best_parameters


## 3.0 Prepare Feature Sets

Min-max scaling is independent for each column and was fitted on fitting records only. Masking after scaling keeps the retained angles identical to the all-feature reference. Both partitions receive the same mask.

In [8]:
assert len(features) == 8
orbital = [i for i, name in enumerate(features) if "orbital" in name.lower()]
nasal = [i for i, name in enumerate(features) if "nasal" in name.lower()]
assert len(orbital) == 6 and len(nasal) == 2
assert set(orbital).isdisjoint(nasal)
feature_sets = {"Orbital only": orbital, "Nasal only": nasal}

def masked_angles(inputs, retained):
    masked = np.zeros_like(np.asarray(inputs))
    masked[:, retained] = np.asarray(inputs)[:, retained]
    return pnp.array(masked, requires_grad=False)

for name, retained in feature_sets.items():
    print(name, [features[i] for i in retained])


Orbital only ['Ec-Dc Orbital Width', 'REc-RDc Right Orbital Width', 'LEc-LDc Left Orbital Width', 'UMo-Lmo Orbital Height', 'RUMo-RLMo Right Orbital Height', 'LUMo-LLMo Left Orbital Height']
Nasal only ['RAI-LAI Nasal Width', 'Rh-Ns Nasal Height']


## 4.0 Run the Comparison

Reuse notebook 08 as the all-eight reference: sigmoid, learning rate 0.01, batch size 32, maximum 30 epochs, patience 5 and seeds 42, 123, 2026. Save each completed run; a rerun skips complete artifacts with matching settings. Use a new results folder if code or settings change.

In [9]:
import joblib

output = project_folder / "results" / "qcnn_feature_ablation"
output.mkdir(parents=True, exist_ok=True)
reference_path = project_folder / "results" / "qcnn_sigmoid_readout" / "sigmoid_runs.csv"
reference = pd.read_csv(reference_path)
assert sorted(reference["Seed"].tolist()) == sorted(SEEDS)
assert np.allclose(reference["Learning rate"], RATE)
assert reference["Readout"].eq("Sigmoid").all()
reference["Feature set"] = "All eight"
settings = {
    "workbook_hash": workbook_hash,
    "features": features, "feature_sets": feature_sets,
    "fit_rows": fit_indices.tolist(), "validation_rows": val_indices.tolist(),
    "seeds": SEEDS, "rate": RATE, "max_epochs": MAX_EPOCHS,
    "batch_size": BATCH_SIZE, "patience": PATIENCE,
    "reference_hash": hashlib.sha256(reference_path.read_bytes()).hexdigest(),
}
settings_path = output / "settings.json"
if settings_path.exists():
    assert json.loads(settings_path.read_text(encoding="utf-8")) == settings, "Use a new output folder for changed settings."
else:
    settings_path.write_text(json.dumps(settings, indent=2), encoding="utf-8")
joblib.dump(scaler, output / "fitting_scaler.joblib")

results = []
for name, retained in feature_sets.items():
    fit = masked_angles(X_fit, retained)
    validation = masked_angles(X_val, retained)
    for seed in SEEDS:
        stem = name.lower().replace(" ", "_") + f"_seed_{seed}"
        record = output / f"{stem}.json"
        weights_path = output / f"{stem}_weights.npy"
        history_path = output / f"{stem}_history.csv"
        if all(path.is_file() for path in (record, weights_path, history_path)):
            result = json.loads(record.read_text(encoding="utf-8"))
            assert result["Seed"] == seed and result["Feature set"] == name
            print(f"Using saved run: {name}, seed {seed}", flush=True)
        else:
            print(f"Training: {name}, seed {seed}", flush=True)
            result, history, weights = train_readout(seed, "Learned readout", RATE, fit, validation)
            result.update({"Feature set": name, "Readout": "Sigmoid", "Learning rate": RATE})
            np.save(weights_path, weights)
            pd.DataFrame(history).to_csv(history_path, index=False)
            record.write_text(json.dumps(result, indent=2), encoding="utf-8")
        results.append(result)
        print(f"  Macro F1: {result['Validation macro F1']:.4f}", flush=True)
        runs = pd.concat([reference, pd.DataFrame(results)], ignore_index=True)
        runs.to_csv(output / "validation_runs.csv", index=False)


Training: Orbital only, seed 42
  Macro F1: 0.6139
Training: Orbital only, seed 123
  Macro F1: 0.6184
Training: Orbital only, seed 2026
  Macro F1: 0.6162
Training: Nasal only, seed 42
  Macro F1: 0.7223
Training: Nasal only, seed 123
  Macro F1: 0.7139
Training: Nasal only, seed 2026
  Macro F1: 0.7158


## 5.0 Compare Validation Results

Report mean and standard deviation across seeds. Negative paired differences indicate lower macro F1 after removing features. This measures performance of the feature sets under this circuit and split; it does not establish causality.

In [10]:
assert len(runs) == 9 and not runs.duplicated(["Feature set", "Seed"]).any()
metrics = ["Validation macro F1", "Validation accuracy", "Validation loss", "Female recall", "Male recall", "Best epoch", "Runtime (s)"]
summary = runs.groupby("Feature set")[metrics].agg(["mean", "std"])
paired = runs.pivot(index="Seed", columns="Feature set", values="Validation macro F1")
for name in feature_sets:
    paired[f"{name} minus all eight"] = paired[name] - paired["All eight"]
display(summary.round(4))
display(paired.round(4))
summary.to_csv(output / "validation_summary.csv")
paired.to_csv(output / "paired_macro_f1.csv")


Validation macro F1         Validation accuracy         Validation loss         Female recall         Male recall         Best epoch         Runtime (s)         
                            mean     std                mean     std            mean     std          mean     std        mean     std       mean     std        mean      std
Feature set                                                                                                                                                                   
All eight                 0.7241  0.0122              0.7302  0.0137          0.5016  0.0011        0.6712  0.0237      0.7754  0.0370    29.0000  0.0000    129.6667  11.0096
Nasal only                0.7173  0.0044              0.7222  0.0034          0.5156  0.0029        0.6804  0.0209      0.7544  0.0122    29.0000  0.0000    142.5638  14.5717
Orbital only              0.6162  0.0022              0.6369  0.0000          0.6284  0.0008        0.4658  0.0137      0.7684  0.0105    23.3333  5.5076    123.2772  20.0706

Feature set,All eight,Nasal only,Orbital only,Orbital only minus all eight,Nasal only minus all eight
Seed,,,,,
42,0.7326,0.7223,0.6139,-0.1187,-0.0104
123,0.7295,0.7139,0.6184,-0.1111,-0.0156
2026,0.7102,0.7158,0.6162,-0.0940,0.0056
